# Spot the Mask — Exploratory Data Analysis
Understand the dataset before training.


In [ ]:
import sys; sys.path.insert(0, '..')
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('../data/raw/train_labels.csv')
print(f'Shape: {df.shape}')
print(df.head())
print(df['target'].value_counts())

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['target'].value_counts().plot.bar(ax=axes[0], color=['#E8593C','#3B8BD4'])
axes[0].set_title('Class distribution')
axes[0].set_xticklabels(['No mask (0)', 'Mask (1)'], rotation=0)

df['target'].value_counts().plot.pie(
    ax=axes[1], labels=['Mask', 'No mask'],
    autopct='%1.1f%%', colors=['#3B8BD4','#E8593C']
)
axes[1].set_title('Class split')
plt.tight_layout()
plt.savefig('../outputs/logs/class_distribution.png', dpi=120)
plt.show()

In [ ]:
# Image size distribution
images_dir = Path('../data/raw/images')
heights, widths = [], []
for fname in list(images_dir.glob('*'))[:500]:
    img = cv2.imread(str(fname))
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=30, color='#3B8BD4', alpha=0.8)
axes[0].set_title('Image heights'); axes[0].set_xlabel('pixels')
axes[1].hist(widths,  bins=30, color='#E8593C', alpha=0.8)
axes[1].set_title('Image widths'); axes[1].set_xlabel('pixels')
plt.tight_layout()
plt.show()
print(f'Height  — min:{min(heights)}  max:{max(heights)}  mean:{np.mean(heights):.0f}')
print(f'Width   — min:{min(widths)}   max:{max(widths)}   mean:{np.mean(widths):.0f}')

In [ ]:
# Sample images from each class
for label, name in [(1, 'With mask'), (0, 'Without mask')]:
    subset = df[df['target'] == label].sample(5, random_state=42)
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(name, fontsize=14)
    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = cv2.imread(str(images_dir / row['image']))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(row['image'][:12], fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Pixel statistics
means, stds = [], []
sample_files = list(images_dir.glob('*'))[:200]
for fpath in sample_files:
    img = cv2.imread(str(fpath))
    if img is not None:
        img_norm = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        means.append(img_norm.reshape(-1, 3).mean(axis=0))
        stds.append(img_norm.reshape(-1, 3).std(axis=0))
means = np.array(means)
stds  = np.array(stds)
print('Dataset mean (R,G,B):', means.mean(axis=0).round(4))
print('Dataset std  (R,G,B):', stds.mean(axis=0).round(4))
print('ImageNet mean:        [0.485, 0.456, 0.406]')
print('ImageNet std:         [0.229, 0.224, 0.225]')